In [4]:
include("../RayTracing.jl")

Main.RayTracing

In [54]:
parsed_args = RayTracing.parse_commandline()
parsed_args["scene-number"] = 4
    
# set up logging
logger = RayTracing.setup_logging(parsed_args["debug"])
RayTracing.global_logger(logger)

# set random seed
RayTracing.Random.seed!(parsed_args["seed"])

DIM = 22
parsed_args["image-dim"] = DIM

22

In [55]:
I, scene = RayTracing.build_scene(parsed_args)


There are 37 objects in the scene, building BVH
  0.000051 seconds (588 allocations: 49.688 KiB)
Done building BVH
Using 5 samples per pixel
There are 2 lights in the scene


(Main.RayTracing.BDPTIntegrator(Main.RayTracing.PerspectiveCamera(Main.RayTracing.ProjectiveCamera(Main.RayTracing.CameraCore(Main.RayTracing.Transformation([1.0 0.0 0.0 278.0; 0.0 1.0 0.0 278.0; 0.0 0.0 1.0 -800.0; 0.0 0.0 0.0 1.0], [1.0 0.0 0.0 -278.0; 0.0 1.0 0.0 -278.0; 0.0 0.0 1.0 800.0; 0.0 0.0 0.0 1.0]), 0.0, 1.0, Main.RayTracing.Film([22.0, 22.0], Main.RayTracing.Bounds2([0.0, 0.0], [22.0, 22.0]), 0.001, Main.RayTracing.BoxFilter([0.1, 0.1]), "yeehaw.exr", Main.RayTracing.Pixel[Main.RayTracing.Pixel([0.0, 0.0, 0.0], 0.0, Main.RayTracing.AtomicXYZPBRT(Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0))) Main.RayTracing.Pixel([0.0, 0.0, 0.0], 0.0, Main.RayTracing.AtomicXYZPBRT(Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0))) … Main.RayTracing.Pixel([0.0, 0.0, 0.0], 0.0, Main.RayTracing.AtomicXYZPBRT(Base.Threads.Atomic{Float64}(0.0), Base.Threads.Atomic{Float64}(0.0), Bas

In [56]:
# Instantiate a Filter
filter = RayTracing.BoxFilter(RayTracing.Pnt2(.1, .1))

image = zeros(Float64, (DIM, DIM))
for x in 0:(DIM-1)
    for y in 0:(DIM-1)
        xmin = x/DIM
        ymin = y/DIM
        xmax = (x+1)/DIM
        ymax = (y+1)/DIM
        parsed_args["crop-window"] = [xmin, ymin, xmax, ymax]
        println("Working on $([xmin, ymin, xmax, ymax])")

        # Instantiate a Film
        film = RayTracing.Film(
            RayTracing.Pnt2(parsed_args["image-dim"], parsed_args["image-dim"]),
            RayTracing.Bounds2(
                RayTracing.Pnt2(parsed_args["crop-window"][1], parsed_args["crop-window"][2]),
                RayTracing.Pnt2(parsed_args["crop-window"][3], parsed_args["crop-window"][4])),
            filter,
            1.0,
            1.0,
            parsed_args["file-name"]
        )

        # Instantiate a Camera
        look_from = RayTracing.Pnt3(278, 278, -800)
        look_at = RayTracing.Pnt3(278, 278, 0)
        up = RayTracing.Vec3(0, 1, 0)
        screen = RayTracing.Bounds2(RayTracing.Pnt2(-1, -1), RayTracing.Pnt2(1, 1))
        C = RayTracing.PerspectiveCamera(RayTracing.LookAt(look_from, look_at, up), screen, 0.0, 1.0, 0.0, 1e6, 40.0, film)

        # Instantiate a Sampler
        # S = ZSobolSampler(parsed_args["samples-per-pixel"], film.full_resolution, Int8(2))
        S = RayTracing.StratifiedSampler(parsed_args["samples-per-pixel"], parsed_args["jitter"])

        # Instantiate an Integrator
        I = RayTracing.BDPTIntegrator(C, S, parsed_args["max-depth"])

        
        muahaha = @elapsed(RayTracing.render(I, scene, parsed_args, (-1, -1)))
        image[y+1,x+1] = muahaha
    end
end

Working on [0.0, 0.0, 0.045454545454545456, 0.045454545454545456]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.045454545454545456, 0.045454545454545456, 0.09090909090909091]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.09090909090909091, 0.045454545454545456, 0.13636363636363635]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.13636363636363635, 0.045454545454545456, 0.18181818181818182]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.18181818181818182, 0.045454545454545456, 0.22727272727272727]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.22727272727272727, 0.045454545454545456, 0.2727272727272727]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.2727272727272727, 0.045454545454545456, 0.3181818181818182]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.3181818181818182, 0.045454545454545456, 0.36363636363636365]
Rendering 1 tiles
Utilizing 1 threads

Working on [0.0, 0.36363636363636365, 0.045454545454545456

In [57]:
spectrum_img = RayTracing.spectrum_from_float.((image .- minimum(image))./(maximum(image) - minimum(image)))
newimage = zeros(RayTracing.RGB, DIM, DIM)
for y in 1:DIM
    for x in 1:DIM
        newimage[y,x] = RayTracing.RGB(spectrum_img[y,x][1], spectrum_img[y,x][2], spectrum_img[y,x][3])
    end
end
RayTracing.OpenEXR.save("hahaha.exr", newimage)